# Kanoon-Bridge on a Colab T4: dense channel + post-2024 High Court judgments

This notebook does what a CPU laptop can't:

1. Downloads post-2024 **High Court criminal judgments** (eCourts open data, CC-BY-4.0) and measures the **IPC → BNS citation shift** month by month.
2. Adds those judgments to the corpus.
3. Encodes every judgment paragraph with the **multilingual E5 dense model on the GPU**.
4. Tunes the QPP-gated fusion for typed questions on **validation**, retrains both learning-to-rank models, and re-runs **every evaluation**, so the ablation finally has real `+dense` and `+qpp` rows.

**Before you start.** Use Runtime → Change runtime type → **T4 GPU**. Put two things in Google Drive under `MyDrive/kanoon/`:
- `kanoon-bridge.zip` (the project zip)
- the five IL-PCSR parquet files, in `MyDrive/kanoon/ilpcsr/`

**Time:** about 2–2.5 hours. The High Court download takes about 30–60 min, encoding about 15 min, and the evaluation about 1 hour.

**Output:** `MyDrive/kanoon/colab_results.zip`. Unzip it over your project folder.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf kanoon-bridge && unzip -q /content/drive/MyDrive/kanoon/kanoon-bridge.zip -d /content
%cd /content/kanoon-bridge
!mkdir -p data/raw/ilpcsr && cp /content/drive/MyDrive/kanoon/ilpcsr/*.parquet data/raw/ilpcsr/
!ls data/raw/ilpcsr data/raw

In [ ]:
!pip -q install -e ".[dev,dense,hc]"
!python scripts/00_fetch_data.py --only bns
!python scripts/00_fetch_data.py --only procedure
!python -m pytest -q 2>&1 | tail -2

## 1. Post-2024 High Court judgments and the citation shift

`--per-month 30` samples up to 30 criminal judgments per month from each of five High Courts (Delhi, Punjab & Haryana, Allahabad-Lucknow, Kerala, Gujarat), covering 2023–2025. Raise it for a tighter estimate, or lower it if the download is slow. If the run is interrupted, re-running the cell skips files it already has.

In [ ]:
!python scripts/11_hc_judgments.py fetch --years 2023 2024 2025 --per-month 30
!wc -l data/raw/hc/judgments.jsonl

In [ ]:
!python scripts/11_hc_judgments.py shift
from IPython.display import Image
Image('results/figures/citation_shift.png')

In [ ]:
!python scripts/11_hc_judgments.py ingest     # adds them to the corpus (data/processed/hc_docs.jsonl)
!python scripts/11_hc_judgments.py testset    # E10: IPC-numbered queries vs BNS-era judgments

## 2. Build the indexes (with the new judgments)

In [ ]:
!python scripts/01_build_corpus.py && python scripts/02_build_index.py && python scripts/03_build_graph.py

## 3. Dense channel on the GPU

The config asks for `ArkadeepAcharya/NLLB-E5` (Hindi-BEIR). If that model can't be loaded, it falls back to `intfloat/multilingual-e5-base` and prints a message saying so.

In [ ]:
!python scripts/04_encode_dense.py --batch-size 128
!python scripts/apply_tuning.py dense-on
!ls -la data/processed/embeddings

## 4. Tune on validation, retrain both rankers, regenerate the test sets

In [ ]:
!python scripts/09_tune_on_val.py --short --limit 400   # trigram x dense_max grid, typed-question proxies of VAL
!python scripts/apply_tuning.py short
!python scripts/06_train_ltr.py            # whole-judgment queries (val)
!python scripts/06_train_ltr.py --short    # typed questions (train excerpts)
!python scripts/07_make_test_sets.py && python scripts/08_make_judged_sets.py

## 5. Every evaluation (with dense), agreement between judges, figures

In [ ]:
!python scripts/05_run_all_evals.py 2>&1 | grep -v "could not parse"
!python scripts/12_second_judge.py agree
!python scripts/10_report_figures.py

In [ ]:
import pandas as pd
pd.read_csv('results/tables/main_results.csv')[['set','system','P@5','R@10','MAP','nDCG@10','MRR']]

In [ ]:
pd.read_csv('results/tables/ablation_e1_ilpcsr.csv')[['step','MAP','nDCG@10','note']]

In [ ]:
for t in ('ablation_e1q_ilpcsr_short', 'ltr_short', 'agreement', 'citation_shift'):
    try:
        display(t, pd.read_csv(f'results/tables/{t}.csv'))
    except FileNotFoundError:
        print(t, 'missing')

## 6. Save the results to Drive

In [ ]:
!zip -qr /content/drive/MyDrive/kanoon/colab_results.zip results/tables results/figures docs/report/figures \
    configs data/queries data/crosswalk data/processed/index/ltr.json data/processed/index/ltr_short.json \
    data/raw/hc/judgments.jsonl data/processed/hc_docs.jsonl
!ls -la /content/drive/MyDrive/kanoon/
# optional, about 300 MB: the embeddings, if you want the dense channel in the app on your laptop
# !zip -qr /content/drive/MyDrive/kanoon/embeddings.zip data/processed/embeddings